# Decoding Eagle Ford: Why Some Wells Are Hard

This competition is not an abstract curve-matching puzzle — it is **real geosteering**. Every
`typewell` / `horizontal_well` pair is a true-to-life well, and once you know *which* play it is
and *what* the six formation-top columns mean, the whole problem snaps into focus.

This notebook decodes the geology, explains what `TVT` actually is, and shows — with the
competition's own data — **why a minority of wells are genuinely hard**, a fact the academic
geosteering literature predicts and that should shape how you read your own validation scores.

Nothing here is a "solution". It is the **map of the terrain** every solution is built on.


## 1. Which play is this?

Open any `__typewell.csv` and read the `Geology` label column. You will find marker names like
**OLMOS, UPSN (Upson), Anacacho, Austin, Eagle Ford, Buda**. That stratigraphic stack is
unmistakable: this is the **South Texas Eagle Ford play** (Upper Cretaceous, Gulf Coast /
Maverick Basin – San Marcos Arch). The six formation-top columns are the classic Eagle Ford
section, shallow to deep:

| Column | Formation | Age | Gamma-ray character |
|---|---|---|---|
| **ANCC** | Anacacho Ls (top-Austin cap) | Campanian | clean carbonate, low GR |
| **ASTNU** | Austin Chalk — Upper | Coniacian–Campanian | chalk/marl, low–moderate GR |
| **ASTNL** | Austin Chalk — Lower | Turonian–Coniacian | chalk/marl |
| **EGFDU** | Eagle Ford — Upper | Turonian | marl + thin limestone |
| **EGFDL** | Eagle Ford — Lower | Cenomanian | **organic-rich mudrock, HOTTEST GR** (source rock + landing zone) |
| **BUDA** | Buda Limestone | L. Cenomanian | dense limestone, **very low GR (~30 API)** |

Most laterals land in the **Lower Eagle Ford (EGFDL)**; a notable minority target the **Austin
Chalk** — so don't assume every well lands in the same formation.


In [ ]:
import os, glob
import numpy as np, pandas as pd
import matplotlib.pyplot as plt

# Robustly locate the per-well files wherever the competition mounts them.
cands = glob.glob("/kaggle/input/**/*__typewell.csv", recursive=True)
BASE = os.path.dirname(cands[0]) if cands else \
       "/kaggle/input/rogii-wellbore-geology-prediction/train"
wids = sorted({os.path.basename(p).split("__")[0]
               for p in glob.glob(os.path.join(BASE, "*__typewell.csv"))})
print(f"{len(wids)} wells in {BASE}")

def load(wid):
    tw = pd.read_csv(f"{BASE}/{wid}__typewell.csv")
    hw = pd.read_csv(f"{BASE}/{wid}__horizontal_well.csv")
    return tw, hw

WID = wids[0]                      # any clean training well
tw, hw = load(WID)
print("typewell columns  :", list(tw.columns))
print("horizontal columns:", list(hw.columns))


## 2. The gamma-ray backbone

The single most reliable correlation surface in this play is the **Eagle Ford / Buda contact**:
the GR drops from the hot EGFDL shale (often >100, up to ~200 API) to the clean Buda limestone
(~30 API) across a very short interval. That sharp cliff is your anchor. A robust mental model:
parameterise every other marker as an **interval measured from the Buda top**, because the Buda
kick is the least ambiguous feature in the whole log.

Below is a real **type section** (GR vs stratigraphic position TVT) with the six formation tops
annotated. Watch the EGFDL → Buda transition — the GR cliff every correlation engine leans on.


In [ ]:
fig, ax = plt.subplots(figsize=(4.6, 9))
ax.plot(tw["GR"], tw["TVT"], lw=0.8, color="k")
ax.invert_yaxis()                                  # depth increases downward
ax.set_xlabel("Gamma Ray (API)"); ax.set_ylabel("TVT  (ft, stratigraphic)")
ax.set_title(f"Eagle Ford type section\n{WID}")

# formation tops are stored in TVT coordinate -> ~constant per well; take the level
forms = [("ANCC", "#888888"), ("ASTNU", "#33aa77"), ("ASTNL", "#229966"),
         ("EGFDU", "#ee8800"), ("EGFDL", "#dd0000"), ("BUDA", "#0066cc")]
x1 = ax.get_xlim()[1]
for col, color in forms:
    if col in hw.columns and hw[col].notna().any():
        lvl = float(np.nanmedian(hw[col]))
        ax.axhline(lvl, color=color, lw=1.1, ls="--", alpha=.7)
        ax.text(x1, lvl, f"  {col}", color=color, va="center", fontsize=8)
plt.tight_layout(); plt.show()


The competition also ships an official **curtain-section image** per training well
(`{id}.png`) — a great way to build intuition for how the lateral threads the layered geology.


In [ ]:
import matplotlib.image as mpimg
png = f"{BASE}/{WID}.png"
if os.path.exists(png):
    img = mpimg.imread(png)
    plt.figure(figsize=(11, 5)); plt.imshow(img); plt.axis("off")
    plt.title(f"Official curtain-section view — {WID}"); plt.show()
else:
    print("curtain-section png not present for this well; skipping")


## 3. What is `TVT`, really?

Three depth concepts matter:

- **MD** — measured depth along the borehole.
- **TVD / TVDSS** — true vertical depth (from rig floor / from sea level). Always compare two
  wells in **TVDSS**, never raw TVD, or a difference in rig-floor elevation injects a constant bias.
- **TVT (true vertical thickness)** — the *stratigraphic* position: how far you are, vertically,
  below a chosen marker.

Why does the competition ask for **TVT after the Prediction-Start point** instead of TVD? Because
in a landed lateral the bit **tracks bedding** — it holds an almost-flat TVT while the TVD ramps
up and down with structural dip. The small, stationary geological signal lives in **TVT**, so the
natural target is the **residual of TVT over the last-known flat level**, not TVD. The plot below
makes this concrete: TVD swings widely while TVT stays nearly flat.


In [ ]:
md = hw["MD"].to_numpy()
fig, ax = plt.subplots(2, 1, figsize=(10, 6), sharex=True)
if "Z" in hw.columns:
    ax[0].plot(md, hw["Z"].to_numpy(), color="#0066cc")
ax[0].set_ylabel("Z / TVDSS (ft)")
ax[0].set_title(f"{WID}:  TVD (Z) ramps with structural dip ...")
if "TVT" in hw.columns and hw["TVT"].notna().any():
    ax[1].plot(md, hw["TVT"].to_numpy(), color="#dd0000")
    ax[1].set_title("... while TVT (stratigraphic position) stays nearly flat — the target")
else:
    ax[1].set_title("TVT not provided for this well (test wells hide it)")
ax[1].set_ylabel("TVT (ft)"); ax[1].set_xlabel("MD (ft)")
plt.tight_layout(); plt.show()


## 4. Why a minority of wells are genuinely HARD (and it's not your model)

Match the horizontal GR to the typewell GR and **most wells solve cleanly**. But a tail of wells
carries a **per-well datum bias of roughly ±15 ft** — the GR pattern matches **two** stratigraphic
levels almost equally well, and picking the wrong one shifts the whole lateral by a near-constant
offset. The geology explains exactly why:

1. **Milankovitch cyclicity (the dominant cause).** The Eagle Ford / Austin are pervasive
   limestone–marl couplets bundled by orbital eccentricity into ~6 ft and ~20–25 ft super-cycles.
   A matcher can lock onto the wrong-but-near-identical bundle → an "off by ~one bundle" error of
   ~15–30 ft. This is **per-well and non-spatial** (it doesn't cluster geographically).
2. **Small / sub-seismic faults.** South Texas Eagle Ford is cut by NE–SW normal faults; the
   common throw population (15–30 ft) maps straight onto the observed bias on wells that cross one.
3. **Base-Austin unconformity / bentonite mis-ties** — smaller, sub-area effects.

The honest punchline, straight from the geosteering literature: when researchers gave **54 experts
the same well data, their interpretations were inconsistent**. There is no unified quantitative
rule that resolves the ambiguity from GR + geometry alone. So if your validation has a stubborn
tail of high-error wells, that is most likely the **datum ambiguity**, not a bug in your model.
Treat it as a property of the data, carry the alternative as a ranked hypothesis, and don't
overfit to chase it.


## 5. Practical takeaways (play-level, general)

- **Anchor to the Buda top**; parameterise other markers as intervals-from-Buda.
- **Work in TVDSS** across two wells; a constant per-well offset is your friend, not your enemy.
- **Predict the residual** of TVT over the last-known flat level, not TVD.
- **Validate with GroupKFold by well** — the hidden test is *new* wells, so any per-well leakage in
  CV will badly mislead you.
- **Expect a hard tail** driven by the bimodal datum; track your **median** per-well error, not
  just pooled RMSE, to see how close you really are on the easy majority.

If this helped you see the geology behind the numbers, an upvote is appreciated — and corrections
from the petrophysicists in the crowd are very welcome.


---
**More in this series (same competition):**

- [The ±15 ft datum: why honest CV bottoms out](https://www.kaggle.com/code/souldrive/the-15-ft-datum-why-honest-cv-bottoms-out) — the midpoint-optimality math for the genuinely-ambiguous subset, plus a multi-hypothesis (CNN-MTP) test of whether you can beat the average.
- [Is your target data-limited? A 3-test check](https://www.kaggle.com/code/souldrive/is-your-target-data-limited-a-3-test-check) — a reusable, competition-agnostic toolkit to tell a model-limited plateau from a data-limited one.
